# **Computational Drug Discovery [Part 1] Download Bioactivity Data**

Chanin Nantasenamat

[*'Data Professor' YouTube channel*](http://youtube.com/dataprofessor)

In this Jupyter notebook, we will be building a real-life **data science project** that you can include in your **data science portfolio**. Particularly, we will be building a machine learning model using the ChEMBL bioactivity data.

---

## **ChEMBL Database**

The [*ChEMBL Database*](https://www.ebi.ac.uk/chembl/) is a database that contains curated bioactivity data of more than 2 million compounds. It is compiled from more than 76,000 documents, 1.2 million assays and the data spans 13,000 targets and 1,800 cells and 33,000 indications.
[Data as of March 25, 2020; ChEMBL version 26].

## **Installing libraries**

Install the ChEMBL web service package so that we can retrieve bioactivity data from the ChEMBL Database.

In [40]:
! pip install -U chembl_webresource_client

## **Importing libraries**

In [41]:
# Import necessary libraries
import pandas as pd
from chembl_webresource_client.new_client import new_client

## **Search for Target protein**

### **Target search for coronavirus**

In [42]:
# Target search for coronavirus
# it should produce same results as https://www.ebi.ac.uk/chembl/search_results/coronavirus?focusEntity=targets
target = new_client.target
target_query = target.search('coronavirus')
targets = pd.DataFrame.from_dict(target_query)
targets

,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Coronavirus,Coronavirus,17.0,False,CHEMBL613732,[],ORGANISM,11119
1,[],Feline coronavirus,Feline coronavirus,14.0,False,CHEMBL612744,[],ORGANISM,12663
2,[],Murine coronavirus,Murine coronavirus,14.0,False,CHEMBL5209664,[],ORGANISM,694005
3,[],Canine coronavirus,Canine coronavirus,14.0,False,CHEMBL5291668,[],ORGANISM,11153
4,[],Bovine coronavirus,Bovine coronavirus,14.0,False,CHEMBL6066646,[],ORGANISM,11128
5,[],Human coronavirus 229E,Human coronavirus 229E,13.0,False,CHEMBL613837,[],ORGANISM,11137
6,[],Human coronavirus OC43,Human coronavirus OC43,13.0,False,CHEMBL5209665,[],ORGANISM,31631
7,[],unidentified human coronavirus,unidentified human coronavirus,13.0,False,CHEMBL6195448,[],ORGANISM,694448
8,[],Human coronavirus NL63,Human coronavirus NL63,13.0,False,CHEMBL6195756,[],ORGANISM,277944
9,[],Middle East respiratory syndrome-related coron...,Middle East respiratory syndrome-related coron...,9.0,False,CHEMBL4296578,[],ORGANISM,1335626


### **Select and retrieve bioactivity data for *SARS coronavirus 3C-like proteinase* (fifth entry)**

We will assign the fifth entry (which corresponds to the target protein, *coronavirus 3C-like proteinase*) to the ***selected_target*** variable

In [43]:
t_id = 12
selected_target = targets.target_chembl_id[t_id]
selected_target

'CHEMBL4523582'

Here, we will retrieve only bioactivity data for *coronavirus 3C-like proteinase* (CHEMBL3927) that are reported as IC$_{50}$ values in nM (nanomolar) unit.

In [44]:
# it is = https://www.ebi.ac.uk/chembl/explore/activities/STATE_ID:fnJwd_c6pYPq67sEoJC7qw%3D%3D
# Activity records describe experimental measurements between a molecule and a biological target.
activity = new_client.activity
# From those activities, keep only measurements whose standard type is IC50.
# IC50 tells you how much of a drug you need to inhibit a target by 50%. Eg.:
# Drug A: IC50 = 5 nM
# Drug B: IC50 = 500 nM
# Drug A is more potent than Drug B.
# res = activity.filter(target_chembl_id=selected_target).filter(standard_type="IC50")
res = new_client.activity.filter(
    target_chembl_id=selected_target,
    standard_type="IC50",
    standard_relation="=",           # drop censored values at the server
).only(["molecule_chembl_id", "canonical_smiles", "standard_type",
        "standard_value", "standard_units", "assay_description"])
# Just to read it...
print(type(res)) # chembl_webresource_client.query_set.QuerySet

# for r in res:
#     print(r["molecule_chembl_id"], r["standard_value"])

<class 'chembl_webresource_client.query_set.QuerySet'>


In [ ]:
# chembl_webresource_client.query_set.QuerySet is lazy. So download 1 shot in a list (and reuse if needed)
records = list(res)
df = pd.DataFrame.from_dict(records)

In [ ]:
df.head(3)

In [ ]:
# check that IC50 is unique value
df.standard_type.unique()

Finally we will save the resulting bioactivity data to a CSV file **bioactivity_data.csv**.

In [ ]:
df.to_csv('bioactivity_data.csv', index=False)

## **Copying files to Google Drive**

Firstly, we need to mount the Google Drive into Colab so that we can have access to our Google adrive from within Colab.

In [ ]:
#from google.colab import drive
#drive.mount('/content/gdrive/', force_remount=True)


Next, we create a **data** folder in our **Colab Notebooks** folder on Google Drive.

In [ ]:
#! mkdir "/content/gdrive/My Drive/Colab Notebooks/data2"
# ! cp bioactivity_data.csv "/content/gdrive/My Drive/Colab Notebooks/data"
# ! ls -l "/content/gdrive/My Drive/Colab Notebooks/data"

Let's see the CSV files that we have so far.

Taking a glimpse of the **bioactivity_data.csv** file that we've just created.

In [ ]:
! head bioactivity_data.csv

## **Handling missing data**
If any compounds has missing value for the **standard_value** column then drop it

In [ ]:
df2 = df[df.standard_value.notna()]
df2

Apparently, for this dataset there is no missing data. But we can use the above code cell for bioactivity data of other target protein.

## **Data pre-processing of the bioactivity data**

### **Labeling compounds as either being active, inactive or intermediate**
The bioactivity data is in the IC50 unit. Compounds having values of less than 1000 nM will be considered to be **active** while those greater than 10,000 nM will be considered to be **inactive**. As for those values in between 1,000 and 10,000 nM will be referred to as **intermediate**.

In [ ]:
bioactivity_class = []
for i in df2.standard_value:
  print(i)
  if float(i) >= 10000:
    bioactivity_class.append("inactive")
  elif float(i) <= 1000:
    bioactivity_class.append("active")
  else:
    bioactivity_class.append("intermediate")

### **Iterate the *molecule_chembl_id* to a list**

In [ ]:
mol_cid = []
for i in df2.molecule_chembl_id:
  print(i)
  mol_cid.append(i)

### **Iterate *canonical_smiles* to a list**

In [ ]:
canonical_smiles = []
for i in df2.canonical_smiles:
  print(i)
  canonical_smiles.append(i)

### **Iterate *standard_value* to a list**

In [ ]:
standard_value = []
for i in df2.standard_value:
  print(i)
  standard_value.append(i)

### **Combine the 4 lists into a dataframe**

In [ ]:
data_tuples = list(zip(mol_cid, canonical_smiles, bioactivity_class, standard_value))
df3 = pd.DataFrame( data_tuples,  columns=['molecule_chembl_id', 'canonical_smiles', 'bioactivity_class', 'standard_value'])

In [ ]:
df3

### **Alternative method**

In [ ]:
# selection = ['molecule_chembl_id', 'canonical_smiles', 'standard_value']
# df3 = df2[selection]
# pd.concat([df3,pd.Series(bioactivity_class)], axis=1)
# df3

Saves dataframe to CSV file

In [ ]:
df3.to_csv('bioactivity_preprocessed_data.csv', index=False)

In [ ]:
! ls -l

Let's copy to the Google Drive

In [ ]:
#! cp bioactivity_preprocessed_data.csv "/content/gdrive/My Drive/Colab Notebooks/data"
# ! ls "/content/gdrive/My Drive/Colab Notebooks/data"

---